# 🐾 Deep Learning in the Kalahari · Session 3 of 3
## *Understanding Deep Learning*, chapters 7–8: backprop and performance

**Transition Bio · ML Reading Group**

This series works through chapters 1–8 of Simon J.D. Prince's **_Understanding Deep Learning_** (MIT Press; free at [udlbook.github.io](https://udlbook.github.io/udlbook/)) in three sessions. Each one is about **20 min of slides and 60–70 min of hands-on puzzles**. We follow the book's notation, square brackets and all: $\mathbf{y} = \mathbf{f}[\mathbf{x}, \boldsymbol\phi]$.

| Session | Topic | Chapters |
|---|---|---|
| Session 1 | Models | Ch 1–4 |
| Session 2 | Losses and training | Ch 5–6 |
| **Session 3** | Backprop and performance | Ch 7–8 |

---

### 🦦 The story
You've just joined a long-running field station in the Kalahari that studies a wild meerkat mob. The mob is about thirty animals, run by a dominant female (in meerkat society, she's in charge). The station has a lot of data and no deep-learning expert, so you're it. Every chapter sets you a new problem from the field.

All datasets are **synthetic**. They're generated in this notebook so that we know the ground truth and can check your answers.

### 🧭 How each part works
| Symbol | Meaning |
|---|---|
| 📖 | Recap of the chapter's maths, with the equations derived properly |
| 💻 | We build it **from scratch in NumPy** first; PyTorch only appears once you've earned it |
| 🧩 ⭐ / ⭐⭐ / ⭐⭐⭐ | Puzzle: quick / medium / stretch. Fill in the `TODO`, then run the 🧪 check cell |
| 🔑 | Hidden solution cell. Double-click to open it. Have a go first; it's there when you need it |
| 🏆 | Open challenge with bronze / silver / gold targets to aim for |

> **Important:** later cells use the functions you write. If you get stuck, **run the 🔑 solution cell** (you don't have to read it) so the rest of the notebook still works.

> 📦 **Coming from Sessions 1–2?** The hidden *carry-over* cell after Setup re-creates everything this session needs from earlier parts (e.g. `deep_forward` from Part 4), so you don't need to re-run old notebooks.

### 🗺️ This session (≈ 70 min hands-on)
| | | Suggested time |
|---|---|---|
| Part 7 | **Gradients and initialization** (Ch 7): backprop by hand and whispers down the burrow | ~25 min |
| Part 8 | **Measuring performance** (Ch 8): bias, variance, double descent and the curse of dimensionality | ~30 min |
| Part 9 | **Capstone** (Ch 2–8): from meerkats to molecules (a toy condensate predictor) | ~15 min |

⏱ **Short on time?** Skip the stretch puzzles **7.4, 8.5 and Challenge 8** in the room and finish them afterwards; they're good practice to finish afterwards.

At the end, `progress()` shows which puzzles you've solved and which are left. It's just for you; nothing is recorded or shared.

*Book figures and problem numbers are cited as e.g. "Fig 3.3" and "P3.8".*

In [ ]:
#@title 🐾 Setup: run me first! (plotting style, score keeper, referee)
import math, itertools, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=4, suppress=True)

# Transition Bio palette
TB = dict(green="#008759", lgreen="#73AE63", blue="#1366AB", lblue="#73B0E6",
          purple="#8F64B7", red="#CF1D10", orange="#EF8E00", grey="#6B6B6B", sand="#E9D8B4")
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (6, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[TB["green"], TB["blue"], TB["orange"],
                                         TB["purple"], TB["red"], TB["lblue"], TB["lgreen"]]),
    "font.size": 10.5, "axes.titleweight": "bold"})

# ---------------- progress tracker (private to you; nothing is shared) ----------------
_SOLVED, _PEEKED, _SEEN = set(), set(), set()
SESSION = 3

def peek(pid):
    _PEEKED.add(pid)

def check(pid, stars, test, ok_msg="", hint=""):
    """Run `test` (a zero-arg function returning True/False) and report the result."""
    _SEEN.add(pid)
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"🕳️  Puzzle {pid}: still a TODO. Fill it in and re-run (or run the 🔑 cell).")
        return
    except Exception as e:
        print(f"💥 Puzzle {pid} raised {type(e).__name__}: {e}")
        if hint: print("   💡 Hint:", hint)
        return
    if ok:
        _SOLVED.add(pid)
        print(f"✅ Puzzle {pid} solved!  {ok_msg}")
    else:
        print(f"❌ Puzzle {pid}: not quite.", ("💡 " + hint) if hint else "")

def progress():
    own = sorted(_SOLVED - _PEEKED); helped = sorted(_SOLVED & _PEEKED); todo = sorted(_SEEN - _SOLVED)
    print(f"🐾 Session {SESSION}: {len(_SOLVED)} of {len(_SEEN)} puzzles solved "
          f"({len(own)} on your own, {len(helped)} after opening the solution).")
    if todo: print("   Still to do:", ", ".join(todo))

def close(a, b, tol=1e-6):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def num_grad(fn, params, eps=1e-6):
    """Central finite-difference gradient of scalar fn(params) w.r.t. a 1D array."""
    params = np.array(params, float); g = np.zeros_like(params)
    for i in range(len(params)):
        d = np.zeros_like(params); d[i] = eps
        g[i] = (fn(params + d) - fn(params - d)) / (2 * eps)
    return g

def medal(score, bronze, silver, gold, higher_is_better=False):
    s = score if higher_is_better else -score
    b, si, g = (bronze, silver, gold) if higher_is_better else (-bronze, -silver, -gold)
    if s >= g: return "🥇 GOLD"
    if s >= si: return "🥈 SILVER"
    if s >= b: return "🥉 BRONZE"
    return "🐣 no medal yet: keep digging"

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
print("Setup complete 🐾  | numpy", np.__version__, "| torch available:", HAS_TORCH)

In [ ]:
#@title 📦 Carry-over from Sessions 1–2 (run me: re-creates what Parts 7–9 need)
def relu(z):
    return np.maximum(z, 0.0)

# Part 4: the general deep network (eq 4.15) and He-style initialisation
def deep_forward(x, betas, omegas, act=relu):
    all_f, all_h = [], [x]
    K = len(omegas) - 1
    for k in range(K + 1):
        f = betas[k] + omegas[k] @ all_h[-1]
        all_f.append(f)
        if k < K:
            all_h.append(act(f))
    return all_f[-1], all_f, all_h

def init_params(dims, sigma2=None, rng=np.random.default_rng(0)):
    betas, omegas = [], []
    for d_in, d_out in zip(dims[:-1], dims[1:]):
        s2 = 2.0 / d_in if sigma2 is None else sigma2
        omegas.append(rng.normal(0, np.sqrt(s2), (d_out, d_in))); betas.append(np.zeros((d_out, 1)))
    return betas, omegas
print("Carried over: relu, deep_forward, init_params 🐾")

---
# Part 7 · Gradients and initialization (Chapter 7): 📣 whispers down the burrow
### 📖 Recap: backpropagation = chain rule + caching
**Toy model** (eq 7.5): one hidden unit per layer, with sin, exp and cos as the activations:
$$\mathrm{f}[x,\boldsymbol\phi]=\beta_3+\omega_3\cos\!\Big[\beta_2+\omega_2\exp\big[\beta_1+\omega_1\sin[\beta_0+\omega_0x]\big]\Big],\qquad \ell_i=(\mathrm{f}[x_i,\boldsymbol\phi]-y_i)^2 .$$

**Forward pass** (eq 7.9): compute and *store* $f_0=\beta_0+\omega_0x,\ h_1=\sin f_0,\ f_1=\beta_1+\omega_1h_1,\ h_2=\exp f_1,\ f_2=\beta_2+\omega_2h_2,\ h_3=\cos f_2,\ f_3=\beta_3+\omega_3h_3$.

**Backward pass #1**: work backwards through the intermediate variables. $\frac{\partial\ell}{\partial f_3}=2(f_3-y)$, then $\frac{\partial\ell}{\partial h_3}=\omega_3\frac{\partial\ell}{\partial f_3}$, then $\frac{\partial\ell}{\partial f_2}=-\sin f_2\,\frac{\partial\ell}{\partial h_3}$, and so on.

**Backward pass #2**: the parameter gradients are $\frac{\partial\ell}{\partial\beta_k}=\frac{\partial\ell}{\partial f_k}$ and $\frac{\partial\ell}{\partial\omega_k}=h_k\frac{\partial\ell}{\partial f_k}$, with $h_0=x$.

**Vector form** (eqs 7.24–7.26), for a deep ReLU net:
$$\frac{\partial\ell}{\partial\boldsymbol\beta_k}=\frac{\partial\ell}{\partial\mathbf{f}_k},\qquad \frac{\partial\ell}{\partial\boldsymbol\Omega_k}=\frac{\partial\ell}{\partial\mathbf{f}_k}\mathbf{h}_k^T,\qquad \frac{\partial\ell}{\partial\mathbf{f}_{k-1}}=\mathbb{I}[\mathbf{f}_{k-1}>0]\odot\Big(\boldsymbol\Omega_k^T\frac{\partial\ell}{\partial\mathbf{f}_k}\Big).$$

In [ ]:
# 💻 Toy model forward pass
TOY = dict(beta0=1.0, beta1=2.0, beta2=-0.5, beta3=0.4, omega0=0.1, omega1=-0.4, omega2=2.0, omega3=3.0)
x_toy, y_toy = 2.3, 2.0

def toy_forward(x, p):
    f0 = p["beta0"] + p["omega0"] * x; h1 = np.sin(f0)
    f1 = p["beta1"] + p["omega1"] * h1; h2 = np.exp(f1)
    f2 = p["beta2"] + p["omega2"] * h2; h3 = np.cos(f2)
    f3 = p["beta3"] + p["omega3"] * h3
    return dict(f0=f0, h1=h1, f1=f1, h2=h2, f2=f2, h3=h3, f3=f3)

def toy_loss(x, y, p):
    return (toy_forward(x, p)["f3"] - y) ** 2
print("forward values:", {k: round(v, 4) for k, v in toy_forward(x_toy, TOY).items()}, "| loss", round(toy_loss(x_toy, y_toy, TOY), 4))

### 🧩 Puzzle 7.1 ⭐⭐ Backprop the toy model by hand
Fill in the backward pass. Every line is one multiplication by a local derivative.

In [ ]:
def toy_backward(x, y, p):
    v = toy_forward(x, p)
    # Backward pass #1: derivatives w.r.t. intermediate quantities
    dl_df3 = 2 * (v["f3"] - y)
    dl_dh3 = None   # TODO
    dl_df2 = None   # TODO
    dl_dh2 = None   # TODO
    dl_df1 = None   # TODO
    dl_dh1 = None   # TODO
    dl_df0 = None   # TODO
    # Backward pass #2: derivatives w.r.t. parameters
    grads = dict(beta0=None, omega0=None, beta1=None, omega1=None,      # TODO
                 beta2=None, omega2=None, beta3=None, omega3=None)
    return grads

In [ ]:
# 🧪 Check puzzle 7.1
def _t71():
    g = toy_backward(x_toy, y_toy, TOY)
    if any(v is None for v in g.values()): raise NotImplementedError
    keys = list(TOY)
    def L(vec): return toy_loss(x_toy, y_toy, dict(zip(keys, vec)))
    gn = num_grad(L, [TOY[k] for k in keys])
    bad = [k for k, n in zip(keys, gn) if not np.isclose(g[k], n, rtol=1e-5, atol=1e-6)]
    if bad: print("   mismatch:", bad)
    return not bad
check("7.1", 2, _t71, "Notice how each line reuses the one before it. That reuse is the whole trick.",
      "d sin = cos, d exp = exp (= h2!), d cos = -sin. And dl/domega_k = h_k * dl/df_k, with h_0 = x.")

In [ ]:
#@title 🔑 Solution 7.1 — double-click to peek (have a go first!)
peek('7.1')
def toy_backward(x, y, p):
    v = toy_forward(x, p)
    dl_df3 = 2 * (v["f3"] - y)
    dl_dh3 = p["omega3"] * dl_df3
    dl_df2 = -np.sin(v["f2"]) * dl_dh3
    dl_dh2 = p["omega2"] * dl_df2
    dl_df1 = v["h2"] * dl_dh2            # derivative of exp is exp
    dl_dh1 = p["omega1"] * dl_df1
    dl_df0 = np.cos(v["f0"]) * dl_dh1
    return dict(beta0=dl_df0, omega0=x * dl_df0, beta1=dl_df1, omega1=v["h1"] * dl_df1,
                beta2=dl_df2, omega2=v["h2"] * dl_df2, beta3=dl_df3, omega3=v["h3"] * dl_df3)

### 🧩 Puzzle 7.2 ⭐⭐⭐ Backprop for a deep ReLU network
Use `deep_forward` from Part 4 (re-created in the carry-over cell), which stores `all_f` and `all_h`, and a least squares loss $\ell=\|\mathbf{f}_K-\mathbf{y}\|^2$ for a **single example** (`x` has shape `(Di, 1)`). Implement the three vector equations above. The checker compares against finite differences and, if it's installed, PyTorch autograd.

In [ ]:
def deep_backward(all_f, all_h, omegas, y):
    """Return (dl_dbetas, dl_domegas) as lists matching betas/omegas, for l = ||f_K - y||^2."""
    K = len(omegas) - 1
    dl_dbetas, dl_domegas = [None] * (K + 1), [None] * (K + 1)
    dl_df = 2 * (all_f[-1] - y)
    for k in range(K, -1, -1):
        # TODO: fill dl_dbetas[k], dl_domegas[k]; then (if k > 0) propagate dl_df to layer k-1
        raise NotImplementedError
    return dl_dbetas, dl_domegas

In [ ]:
# 🧪 Check puzzle 7.2
def _t72():
    rr = np.random.default_rng(0); dims = [3, 5, 4, 2]
    betas, omegas = init_params(dims, rng=rr); betas = [b + rr.normal(0, .3, b.shape) for b in betas]
    xin = rr.normal(size=(3, 1)); yt = rr.normal(size=(2, 1))
    out, fs, hs = deep_forward(xin, betas, omegas); db, dO = deep_backward(fs, hs, omegas, yt)
    def L_w(k, i, j, e):
        o2 = [o.copy() for o in omegas]; o2[k][i, j] += e
        return np.sum((deep_forward(xin, betas, o2)[0] - yt) ** 2)
    def L_b(k, i, e):
        b2 = [b.copy() for b in betas]; b2[k][i, 0] += e
        return np.sum((deep_forward(xin, b2, omegas)[0] - yt) ** 2)
    e = 1e-6; ok = True
    for k in range(len(omegas)):
        gO = np.array([[(L_w(k, i, j, e) - L_w(k, i, j, -e)) / (2 * e) for j in range(omegas[k].shape[1])] for i in range(omegas[k].shape[0])])
        gb = np.array([[(L_b(k, i, e) - L_b(k, i, -e)) / (2 * e)] for i in range(betas[k].shape[0])])
        ok &= close(dO[k], gO, 1e-5) and close(db[k], gb, 1e-5)
    if HAS_TORCH and ok:
        tb = [torch.tensor(b, requires_grad=True) for b in betas]; to = [torch.tensor(o, requires_grad=True) for o in omegas]
        h = torch.tensor(xin)
        for k in range(len(to)):
            f = tb[k] + to[k] @ h; h = torch.relu(f) if k < len(to) - 1 else f
        ((h - torch.tensor(yt)) ** 2).sum().backward()
        ok &= all(close(dO[k], to[k].grad.numpy(), 1e-10) for k in range(len(to)))
        print("   matches torch autograd ✔")
    return ok
check("7.2", 3, _t72, "You just wrote loss.backward(). Seriously, that's all it does (for MLPs).",
      "dl_domegas[k] = dl_df @ all_h[k].T ; dl_df = (all_f[k-1] > 0) * (omegas[k].T @ dl_df)")

In [ ]:
#@title 🔑 Solution 7.2 — double-click to peek (have a go first!)
peek('7.2')
def deep_backward(all_f, all_h, omegas, y):
    K = len(omegas) - 1
    dl_dbetas, dl_domegas = [None] * (K + 1), [None] * (K + 1)
    dl_df = 2 * (all_f[-1] - y)
    for k in range(K, -1, -1):
        dl_dbetas[k] = dl_df.sum(axis=1, keepdims=True)
        dl_domegas[k] = dl_df @ all_h[k].T
        if k > 0:
            dl_df = (all_f[k - 1] > 0) * (omegas[k].T @ dl_df)
    return dl_dbetas, dl_domegas

### 📖 Initialization (§7.5): keep the whisper audible
A message whispered down a 50-meerkat burrow either fades to nothing or turns into a scream. Activations and gradients behave the same way as they pass through layers.

With zero biases and weights $\Omega_{ij}\sim\mathcal{N}(0,\sigma^2_\Omega)$, independent of the activations:
$$\mathbb{E}[f'_i]=0,\qquad \sigma^2_{f'}=\sigma^2_\Omega\sum_{j=1}^{D_h}\mathbb{E}[h_j^2]=\sigma^2_\Omega D_h\,\mathbb{E}[h^2]. \qquad\text{(eq 7.30)}$$
If $f$ is symmetric about zero, then the ReLU gives $\mathbb{E}[h^2]=\tfrac12\sigma_f^2$, so $\sigma^2_{f'}=\tfrac12D_h\sigma^2_\Omega\,\sigma_f^2$ (eq 7.31). Choosing
$$\boxed{\sigma^2_\Omega=\frac{2}{D_h}}\qquad\textbf{(He initialization, eq 7.32)}$$
keeps the variance constant from layer to layer. In the backward pass the same argument uses $D_{h'}$ (eq 7.33), and a compromise between the two is $\sigma^2_\Omega=\frac{4}{D_h+D_{h'}}$ (eq 7.34).

### 🧩 Puzzle 7.3 ⭐⭐ ReLU moments (book P7.14)
For $z\sim\mathcal{N}(0,\sigma^2)$ and $h=\mathrm{ReLU}[z]$, derive $\mathbb{E}[h]$ and $\mathbb{E}[h^2]$ in closed form. (Hint: $\int_0^\infty z\,\mathcal{N}(z;0,\sigma^2)\,dz=\sigma/\sqrt{2\pi}$, and by symmetry half of $\mathbb{E}[z^2]$ comes from $z>0$.)

In [ ]:
def relu_moments(sigma):
    """Return (E[h], E[h^2]) for h = ReLU(z), z ~ N(0, sigma^2)."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 7.3
def _t73():
    z = np.random.default_rng(0).normal(0, 1.7, 2_000_000); h = relu(z)
    m1, m2 = relu_moments(1.7); print(f"   MC: E[h]={h.mean():.4f}, E[h²]={np.mean(h**2):.4f}   yours: {m1:.4f}, {m2:.4f}")
    return np.isclose(m1, 1.7 / np.sqrt(2 * np.pi), rtol=1e-9) and np.isclose(m2, 1.7 ** 2 / 2, rtol=1e-9)
check("7.3", 2, _t73, "The factor 1/2 in He init comes straight from E[h^2] = sigma^2 / 2.",
      "E[h] = sigma/sqrt(2*pi), E[h^2] = sigma^2/2")

In [ ]:
#@title 🔑 Solution 7.3 — double-click to peek (have a go first!)
peek('7.3')
def relu_moments(sigma):
    return sigma / np.sqrt(2 * np.pi), sigma ** 2 / 2

In [ ]:
# 💻 Recreate Fig 7.7: 50 layers x 100 units, five weight variances
def init_experiment(sigma2, K=50, D=100, seed=0):
    rr = np.random.default_rng(seed)
    betas, omegas = init_params([D] + [D] * K + [1], sigma2=sigma2, rng=rr)
    xin = rr.normal(size=(D, 1))
    out, fs, hs = deep_forward(xin, betas, omegas)
    db, dO = deep_backward(fs, hs, omegas, np.zeros((1, 1)))
    fwd = [np.var(h) for h in hs[1:]]
    bwd = [np.var(g) for g in dO[:-1]]
    return fwd, bwd

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s2, col in zip([0.001, 0.01, 0.02, 0.1, 1.0], [TB["blue"], TB["lblue"], TB["green"], TB["orange"], TB["red"]]):
    fwd, bwd = init_experiment(s2)
    ax[0].semilogy(fwd, color=col, lw=2, label=fr"$\sigma^2_\Omega$={s2}")
    ax[1].semilogy(bwd, color=col, lw=2, label=fr"$\sigma^2_\Omega$={s2}")
ax[0].set_title("forward: variance of activations"); ax[0].set_xlabel("layer")
ax[1].set_title("backward: variance of weight gradients"); ax[1].set_xlabel("layer")
ax[0].legend(); plt.show()
print("Only σ² = 2/D_h = 0.02 (He) keeps both curves flat. Everything else vanishes or explodes exponentially.")

### 🧩 Puzzle 7.4 ⭐ Predict the explosion
The input $\mathbf{x}$ has $D_i$ standard-normal entries, all biases are zero, every layer has $D_h$ units, and every weight has variance $\sigma^2_\Omega$. Write the variance of the pre-activations $\mathbf{f}_k$ (with $k=0$ being the first layer, which has no ReLU on its input) in closed form. The checker compares it with a simulation.

In [ ]:
def predicted_preact_var(k, sigma2, Di, Dh):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 7.4
def _t74():
    rr = np.random.default_rng(1); Di = Dh = 200; s2 = 0.013; k = 6
    sims = []
    for _ in range(60):
        b, o = init_params([Di] + [Dh] * 7 + [1], sigma2=s2, rng=rr)
        sims.append(np.var(deep_forward(rr.normal(size=(Di, 1)), b, o)[1][k]))
    pred = predicted_preact_var(k, s2, Di, Dh); print(f"   simulated {np.mean(sims):.4f}  vs  predicted {pred:.4f}")
    return np.isclose(pred, Di * s2 * (Dh * s2 / 2) ** k) and abs(np.mean(sims) / pred - 1) < 0.15
check("7.4", 1, _t74, "A geometric series: stable only when Dh*sigma2/2 = 1.",
      "var(f_0) = Di*sigma2*1; each later layer multiplies by Dh*sigma2/2")

In [ ]:
#@title 🔑 Solution 7.4 — double-click to peek (have a go first!)
peek('7.4')
def predicted_preact_var(k, sigma2, Di, Dh):
    return Di * sigma2 * (Dh * sigma2 / 2) ** k

### 💻 The book's training loop (Fig 7.8), in PyTorch
Everything we built by hand now fits in about 20 lines: He initialization, an MSE loss, SGD with momentum, a step learning-rate schedule, and `loss.backward()` doing Puzzle 7.2 for us.

In [ ]:
if HAS_TORCH:
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    torch.manual_seed(0)
    D_i, D_k, D_o = 10, 40, 5
    model = nn.Sequential(nn.Linear(D_i, D_k), nn.ReLU(), nn.Linear(D_k, D_k), nn.ReLU(), nn.Linear(D_k, D_o))
    def weights_init(layer_in):                                # He initialization
        if isinstance(layer_in, nn.Linear):
            nn.init.kaiming_normal_(layer_in.weight); layer_in.bias.data.fill_(0.0)
    model.apply(weights_init)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    x = torch.randn(100, D_i); y = torch.randn(100, D_o)       # random data, as in the book
    data_loader = DataLoader(TensorDataset(x, y), batch_size=10, shuffle=True)
    hist = []
    for epoch in range(100):
        epoch_loss = 0.0
        for x_batch, y_batch in data_loader:
            optimizer.zero_grad(); loss = criterion(model(x_batch), y_batch)
            loss.backward(); optimizer.step(); epoch_loss += loss.item()
        hist.append(epoch_loss); scheduler.step()
    plt.semilogy(hist, color=TB["green"]); plt.xlabel("epoch"); plt.ylabel("training loss")
    plt.title("Fitting 100 random targets (lr 0.01, halved every 10 epochs)"); plt.show()

---
# Part 8 · Measuring performance (Chapter 8): 🎙️ the meerkat call classifier
### 📖 Recap
- **Generalization** is measured on a separate **test set**. Training error can reach zero while test error stays high.
- For least squares regression, the expected test error splits *exactly* into three parts (eq 8.7):
$$\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big]=\underbrace{\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x,\boldsymbol\phi[\mathcal{D}]]-\mathrm{f}_\mu[x]\big)^2\Big]}_{\text{variance}}+\underbrace{\big(\mathrm{f}_\mu[x]-\mu[x]\big)^2}_{\text{bias}^2}+\underbrace{\sigma^2}_{\text{noise}}$$
  where $\mathrm{f}_\mu[x]=\mathbb{E}_{\mathcal{D}}\big[\mathrm{f}[x,\boldsymbol\phi[\mathcal{D}]]\big]$ is the average fitted model over training sets.
- **Variance** falls with more data. **Bias** falls with more capacity, which usually raises variance: the **bias–variance trade-off**.
- **Double descent**: as capacity grows past the point where the model can memorise the training data, test error can fall *again*.
- The **curse of dimensionality** means high-dimensional training data are extremely sparse, so behaviour *between* data points (the inductive bias) matters.
- Choose **hyperparameters** on a **validation** set, and touch the test set only once.

#### Derivation sketch (eqs 8.2–8.3)
Add and subtract $\mu[x]$: $\;(\mathrm{f}-y)^2=(\mathrm{f}-\mu)^2+2(\mathrm{f}-\mu)(\mu-y)+(\mu-y)^2$. Taking $\mathbb{E}_y$ kills the cross term because $\mathbb{E}_y[y]=\mu$. Repeat the trick with $\mathrm{f}_\mu$ and $\mathbb{E}_{\mathcal{D}}$ to split $(\mathrm{f}-\mu)^2$ into variance plus bias².

In [ ]:
# 💻 The meerkat call dataset: an MNIST-1D-style benchmark (Greydanus 2020), meerkat edition.
CALLS = ["eagle alarm", "jackal alarm", "snake alarm", "close call", "lead call",
         "moving call", "sentinel song", "recruitment", "pup begging", "social chirp"]

def make_meerkat_calls(n, seed=0, noise=0.75, shift=12):
    tr = np.random.default_rng(1234)                             # fixed templates (same species, same repertoire!)
    templates = np.cumsum(tr.normal(0, 1, (10, 12)), axis=1); templates -= templates.mean(1, keepdims=True)
    templates /= np.abs(templates).max(1, keepdims=True)
    r = np.random.default_rng(seed); y = r.integers(0, 10, n); X = np.zeros((n, 40))
    t40 = np.linspace(0, 1, 40)
    for i in range(n):
        tpl = templates[y[i]]; stretch = r.uniform(0.75, 1.25)
        tt = np.clip((t40 - 0.5) * stretch + 0.5, 0, 1)
        sig = np.interp(tt, np.linspace(0, 1, 12), tpl) * r.uniform(0.7, 1.3)
        sig = np.roll(sig, r.integers(-shift, shift + 1))
        sig += r.uniform(-0.6, 0.6) * (t40 - 0.5)                                 # slope (background wind)
        sig += noise * np.convolve(r.normal(size=44), np.ones(5) / 5, "valid")    # correlated noise
        X[i] = sig + 0.15 * noise * r.normal(size=40)
    return X.astype(np.float32), y

Xtr, ytr = make_meerkat_calls(4000, seed=1); Xte, yte = make_meerkat_calls(1000, seed=2)
fig, axs = plt.subplots(2, 5, figsize=(13, 4), sharex=True, sharey=True)
for k, ax in enumerate(axs.ravel()):
    for j in np.flatnonzero(ytr == k)[:4]: ax.plot(Xtr[j], lw=1)
    ax.set_title(CALLS[k], fontsize=9)
plt.suptitle("Four examples of each call type (40 samples each)", fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# 💻 Train the §8.1 model: 40 → 100 → 100 → 10, SGD, lr 0.1, batch 100, 6000 steps
def train_mlp(Xtr, ytr, Xev, yev, width=100, depth=2, lr=0.1, steps=6000, batch=100, seed=0, opt="sgd", log_every=100):
    torch.manual_seed(seed)
    layers, d = [], Xtr.shape[1]
    for _ in range(depth): layers += [torch.nn.Linear(d, width), torch.nn.ReLU()]; d = width
    net = torch.nn.Sequential(*layers, torch.nn.Linear(d, 10))
    for m in net:
        if isinstance(m, torch.nn.Linear): torch.nn.init.kaiming_normal_(m.weight); m.bias.data.zero_()
    o = torch.optim.SGD(net.parameters(), lr=lr) if opt == "sgd" else torch.optim.Adam(net.parameters(), lr=lr)
    Xt, yt, Xe, ye = map(torch.tensor, (Xtr, ytr, Xev, yev)); lossf = torch.nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed); hist = []
    for s in range(steps + 1):
        if s % log_every == 0:
            with torch.no_grad():
                ot, oe = net(Xt), net(Xe)
                hist.append((s, (ot.argmax(1) != yt).float().mean().item(), (oe.argmax(1) != ye).float().mean().item(),
                             lossf(ot, yt).item(), lossf(oe, ye).item()))
        if s == steps: break
        idx = torch.randint(0, len(Xt), (batch,), generator=g)
        o.zero_grad(); lossf(net(Xt[idx]), yt[idx]).backward(); o.step()
    return net, np.array(hist)

if HAS_TORCH:
    t0 = time.time(); _, H = train_mlp(Xtr, ytr, Xte, yte); print(f"trained in {time.time()-t0:.1f}s")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(H[:, 0], 100 * H[:, 1], label="train", color=TB["green"]); ax[0].plot(H[:, 0], 100 * H[:, 2], label="test", color=TB["orange"])
    ax[0].set_ylabel("% error"); ax[0].set_xlabel("step"); ax[0].legend(); ax[0].set_title("Error (cf. Fig 8.2a)")
    ax[1].plot(H[:, 0], H[:, 3], label="train", color=TB["green"]); ax[1].plot(H[:, 0], H[:, 4], label="test", color=TB["orange"])
    ax[1].set_xlabel("step"); ax[1].legend(); ax[1].set_title("Loss (cf. Fig 8.2b)"); plt.show()
    print(f"final train error {100*H[-1,1]:.1f}%   test error {100*H[-1,2]:.1f}%   (chance = 90%)")

In [ ]:
# 🧩 Puzzle 8.1 ⭐  The test LOSS climbs late in training while the test ERROR stays flat. Why?
#  (a) the learning rate is too high, so the optimiser diverges
#  (b) the model keeps making the same mistakes but with growing confidence (softmax logits pushed to extremes)
#  (c) the test set is mislabelled
#  (d) ReLUs die during training
answer_8_1 = None   # TODO "a" / "b" / "c" / "d"

In [ ]:
# 🧪 Check puzzle 8.1
def _t81():
    if answer_8_1 is None: raise NotImplementedError
    return answer_8_1.strip().lower() == "b"
check("8.1", 1, _t81, "Confidently wrong predictions get punished by the log in the cross-entropy (see Part 5B).")

In [ ]:
#@title 🔑 Solution 8.1 — double-click to peek (have a go first!)
peek('8.1')
answer_8_1 = "b"

### 📖 Noise, bias and variance, measured exactly (§8.2)
To *measure* the three terms we need to know the truth, so we go back to 1D regression. The true function is $\mu[x]=\exp(\sin(2\pi x))$, the noise has $\sigma=0.3$, and each dataset has 15 points (one per equal segment of $[0,1]$). The **simplified network** of Fig 8.4 fixes the joints at $0,\frac1D,\dots,\frac{D-1}{D}$, so only the output layer is learned. That makes it **linear in its parameters**, so we can fit it in closed form (book P8.3):
$$\mathbf{A}=\big[\mathbf{1},\ \mathrm{ReLU}[\mathbf{x}-0],\ \mathrm{ReLU}[\mathbf{x}-\tfrac1D],\dots\big],\qquad \hat{\boldsymbol\phi}=(\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T\mathbf{y}.$$

In [ ]:
def mu_true(x): return np.exp(np.sin(2 * np.pi * x))
SIGMA = 0.3
def make_reg_data(n, r):
    x = (np.arange(n) + r.uniform(size=n)) / n          # one sample per equal segment
    return x, mu_true(x) + r.normal(0, SIGMA, n)

In [ ]:
def design(x, D):
    """[1, ReLU(x - 0/D), ReLU(x - 1/D), ..., ReLU(x - (D-1)/D)] -> shape (len(x), D+1)"""
    # TODO
    raise NotImplementedError

def fit_fixed_joints(x, y, D):
    """Least squares coefficients (D+1,). Use np.linalg.lstsq for numerical safety."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 8.2
def _t82():
    r = np.random.default_rng(0); x, y = make_reg_data(15, r)
    A = design(x, 3); c = fit_fixed_joints(x, y, 3)
    ref = np.column_stack([np.ones(15)] + [np.maximum(x - k / 3, 0) for k in range(3)])
    return A.shape == (15, 4) and close(A, ref) and close(c, np.linalg.lstsq(ref, y, rcond=None)[0], 1e-8)
check("8.2", 2, _t82, "Fixing the hidden layer makes the problem linear in its parameters, so it's convex and solvable in closed form.",
      "np.column_stack([np.ones_like(x)] + [relu(x - k/D) for k in range(D)])")

In [ ]:
#@title 🔑 Solution 8.2 — double-click to peek (have a go first!)
peek('8.2')
def design(x, D):
    return np.column_stack([np.ones_like(x)] + [relu(x - k / D) for k in range(D)])
def fit_fixed_joints(x, y, D):
    return np.linalg.lstsq(design(x, D), y, rcond=None)[0]

### 🧩 Puzzle 8.3 ⭐⭐⭐ Estimate bias² and variance by brute force
For a given $D$: draw `n_datasets` training sets, fit each one, and evaluate the fits on a grid `xg = np.linspace(0, 1, 200)`.
- $\mathrm{f}_\mu$ = the mean of the fits over datasets.
- **variance** = the variance across datasets, averaged over the grid.
- **bias²** = $(\mathrm{f}_\mu-\mu)^2$, averaged over the grid.

The checker draws *fresh* test data and verifies eq 8.7: **bias² + variance + σ² ≈ measured test MSE**.

In [ ]:
def bias_variance(D, n_datasets=300, n_points=15, seed=0):
    """Return (bias2, variance), each averaged over xg = np.linspace(0, 1, 200)."""
    r = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 8.3
def _t83():
    D = 3; b2, var = bias_variance(D, 400, 15, seed=1)
    r = np.random.default_rng(99); errs = []
    for _ in range(400):
        x, y = make_reg_data(15, r); c = fit_fixed_joints(x, y, D)
        xt, yt = r.uniform(size=200), None; yt = mu_true(xt) + r.normal(0, SIGMA, 200)
        errs.append(np.mean((design(xt, D) @ c - yt) ** 2))
    pred = b2 + var + SIGMA ** 2; meas = np.mean(errs)
    print(f"   bias²={b2:.4f}  var={var:.4f}  noise={SIGMA**2:.4f}  →  sum={pred:.4f}   measured test MSE={meas:.4f}")
    return abs(pred / meas - 1) < 0.06
check("8.3", 3, _t83, "Eq 8.7, confirmed by simulation.",
      "fits = np.array([design(xg, D) @ fit_fixed_joints(*make_reg_data(n_points, r), D) for _ in range(n_datasets)])")

In [ ]:
#@title 🔑 Solution 8.3 — double-click to peek (have a go first!)
peek('8.3')
def bias_variance(D, n_datasets=300, n_points=15, seed=0):
    r = np.random.default_rng(seed); xg = np.linspace(0, 1, 200)
    fits = np.array([design(xg, D) @ fit_fixed_joints(*make_reg_data(n_points, r), D) for _ in range(n_datasets)])
    f_mu = fits.mean(0)
    return np.mean((f_mu - mu_true(xg)) ** 2), np.mean(fits.var(0))

In [ ]:
# 💻 The bias-variance trade-off (Fig 8.9)
Ds = np.arange(1, 13); bv = np.array([bias_variance(D, 300, 15) for D in Ds])
plt.plot(Ds, bv[:, 0], "o-", color=TB["orange"], label="bias²"); plt.plot(Ds, bv[:, 1], "o-", color=TB["blue"], label="variance")
plt.plot(Ds, bv.sum(1), "--", color=TB["grey"], label="bias² + variance")
best = Ds[np.argmin(bv.sum(1))]; plt.axvline(best, color=TB["green"], ls=":", label=f"best capacity D={best}")
plt.yscale("log"); plt.xlabel("hidden units D (= linear regions)"); plt.legend(); plt.title("Bias-variance trade-off, 15 points"); plt.show()

### 📖 Double descent (§8.4)
Below we use a shallow net with a **random, frozen** first layer (random ReLU features) and fit the output weights with the **minimum-norm** least squares solution (`np.linalg.lstsq`). Minimum norm is a simple stand-in for the implicit regularisation the book suspects SGD provides. We have 30 noisy training points and sweep the width from 1 to 2,000 hidden units, tracking both training and test error. We call the width where training MSE first drops below $\sigma^2/10$ the **(near-)interpolation threshold**: past it, the model has essentially memorised the noise.

In [ ]:
def random_feature_curve(widths, n_train=30, n_test=500, n_rep=40, seed=0):
    r = np.random.default_rng(seed); te = np.zeros((n_rep, len(widths))); tr = np.zeros_like(te)
    for rep in range(n_rep):
        xtr, ytr_ = make_reg_data(n_train, r); xts = r.uniform(size=n_test); yts = mu_true(xts) + r.normal(0, SIGMA, n_test)
        c = r.uniform(-0.1, 1.1, 2000)                                        # joint positions
        a = np.abs(r.normal(0, 1, 2000)) * r.choice([-1, 1], 2000); b = -a * c # frozen first layer
        for j, D in enumerate(widths):
            feats = lambda x: np.column_stack([np.ones_like(x), relu(a[:D] * x[:, None] + b[:D])])
            w = np.linalg.lstsq(feats(xtr), ytr_, rcond=None)[0]               # minimum-norm when under-determined
            te[rep, j] = np.mean((feats(xts) @ w - yts) ** 2); tr[rep, j] = np.mean((feats(xtr) @ w - ytr_) ** 2)
    return np.median(tr, 0), np.median(te, 0)

widths = np.unique(np.round(np.logspace(0, np.log10(2000), 45)).astype(int))
tr_err, te_err = random_feature_curve(widths)
D_interp = widths[np.argmax(tr_err < 0.1 * SIGMA ** 2)]       # first width that (almost) memorises the training set
plt.loglog(widths, te_err, "o-", color=TB["purple"], ms=4, label="test")
plt.loglog(widths, np.maximum(tr_err, 1e-4), "o-", color=TB["green"], ms=3, label="train (floored at 1e-4)")
plt.axvline(D_interp, color=TB["grey"], ls=":"); plt.text(D_interp * 1.15, 2e-3, "(near-)interpolation\nthreshold", color=TB["grey"])
plt.axhline(SIGMA ** 2, color=TB["orange"], ls="--", lw=1); plt.text(150, SIGMA ** 2 * 0.62, "noise floor σ²", color=TB["orange"])
plt.xlabel("hidden units D"); plt.ylabel("median MSE"); plt.legend(loc="lower left"); plt.title("Double descent with random ReLU features"); plt.show()
print(f"training MSE first drops below σ²/10 at D = {D_interp} | test-error peak at D = {widths[np.argmax(te_err)]}")
print(f"classical sweet spot: {te_err[widths < D_interp].min():.3f} | peak: {te_err.max():.2f} | D=2000: {te_err[-1]:.3f}")
print("In this tiny 1D problem the second descent recovers but doesn't beat the classical sweet spot.\n"
      "On MNIST-1D with label noise (Fig 8.10b) it does. How much it helps depends on data, noise and inductive bias.")

In [ ]:
# 🧩 Puzzle 8.4 ⭐  Look at the plot. The test-error peak sits...
#  (a) at D = 1, where the model is too simple
#  (b) around the width where the model can first (almost) memorise the training data (the interpolation threshold)
#  (c) at the largest width, where there are the most parameters
#  (d) at D = 30 exactly, because there are 30 data points (no more, no less)
answer_8_4 = None   # TODO

In [ ]:
# 🧪 Check puzzle 8.4
def _t84():
    if answer_8_4 is None: raise NotImplementedError
    return answer_8_4.strip().lower() == "b"
check("8.4", 1, _t84, "Not exactly 30 here: random joints rarely land in every gap between data points, so the net needs more units than data points to memorise them.",
      "Compare the dotted line with the purple peak.")

In [ ]:
#@title 🔑 Solution 8.4 — double-click to peek (have a go first!)
peek('8.4')
answer_8_4 = "b"

### 🧩 Puzzle 8.5 ⭐⭐ The curse of dimensionality (§8.4.1)
1. Quantise each of the call dataset's 40 input dimensions into 10 bins. How many **bins per training example** are there with 10,000 examples? (The book's number.)
2. A hypersphere of diameter 1 sits inside the unit hypercube. Implement `sphere_fraction(d)`, the fraction of the cube's volume inside the sphere: $V_d(r)=\dfrac{\pi^{d/2}}{\Gamma(d/2+1)}r^d$ with $r=\tfrac12$.

In [ ]:
bins_per_example = None     # TODO

def sphere_fraction(d):
    # TODO (math.gamma)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 8.5
def _t85():
    if bins_per_example is None: raise NotImplementedError
    ok = np.isclose(bins_per_example, 1e36) and np.isclose(sphere_fraction(2), np.pi / 4) and np.isclose(sphere_fraction(3), np.pi / 6)
    print(f"   fraction inside the sphere: d=2 {sphere_fraction(2):.3f}, d=10 {sphere_fraction(10):.4f}, d=40 {sphere_fraction(40):.2e}")
    return ok
check("8.5", 2, _t85, "In 40-D, essentially ALL the volume is in the corners. Your intuition from 3-D does not apply.",
      "10**40 / 10**4; math.pi**(d/2) / math.gamma(d/2 + 1) * 0.5**d")

In [ ]:
#@title 🔑 Solution 8.5 — double-click to peek (have a go first!)
peek('8.5')
bins_per_example = 10 ** 40 / 10 ** 4
def sphere_fraction(d):
    return math.pi ** (d / 2) / math.gamma(d / 2 + 1) * 0.5 ** d

In [ ]:
# 💻 Distances concentrate in high dimensions: nearest and farthest neighbours look alike
r = np.random.default_rng(0); dims = [1, 2, 5, 10, 40, 100, 1000]; ratios = []
for d in dims:
    P = r.uniform(size=(1000, d)); q = r.uniform(size=d); dist = np.linalg.norm(P - q, axis=1)
    ratios.append(dist.max() / dist.min())
plt.semilogx(dims, ratios, "o-", color=TB["blue"]); plt.yscale("log")
plt.xlabel("dimension"); plt.ylabel("farthest / nearest distance"); plt.title("Everything is far from everything"); plt.show()

### 🏆 Challenge 8 ⭐⭐⭐ The Meerkat Call Classifier Cup (§8.5: choosing hyperparameters)
Split the 4,000 training calls into **3,000 train / 1,000 validation**, choose hyperparameters using the **validation** error *only*, and then look at the test set **once**. Medals are awarded on validation accuracy:
- 🥉 > 50%
- 🥈 > 54%
- 🥇 > 56.5%

(The out-of-the-box §8.1 model scores about 53%. A noisy 10-class problem with only 3,000 examples is *hard*.)

Things to play with: `width`, `depth`, `lr`, `steps`, `batch`, and `opt` (`"sgd"` or `"adam"`). Regularisation (chapter 9) is off-limits, though early stopping is *kind of* allowed: `steps` is a hyperparameter!

In [ ]:
Xa, ya, Xv, yv = Xtr[:3000], ytr[:3000], Xtr[3000:], ytr[3000:]
my_hparams = dict(width=100, depth=2, lr=0.1, steps=2000, batch=100, opt="sgd")   # TODO: search!

In [ ]:
# 🧪 Check puzzle C8
def _tc8():
    if not HAS_TORCH: print("   (torch missing)"); return False
    net, H = train_mlp(Xa, ya, Xv, yv, log_every=max(1, my_hparams["steps"]), **my_hparams)
    acc = 1 - H[-1, 2]; print(f"   validation accuracy {100*acc:.1f}% → {medal(acc, .50, .54, .565, higher_is_better=True)}")
    with torch.no_grad(): test_acc = (net(torch.tensor(Xte)).argmax(1).numpy() == yte).mean()
    print(f"   🔒 test accuracy (look once!): {100*test_acc:.1f}%")
    return acc > 0.565
check("C8", 3, _tc8, "Gold! The station will use your classifier on next season's recordings.",
      "Try Adam (lr ~1e-3), width 256-512, depth 2-3 and ~1000 steps. Watch how val error moves with steps.")

In [ ]:
#@title 🔑 Solution C8 — double-click to peek (have a go first!)
peek('C8')
# A quick validation sweep (honest: only validation numbers are used to choose)
best = None
for width, depth, lr, steps in [(256, 2, 1e-3, 1500), (512, 3, 1e-3, 1000)]:
    hp = dict(width=width, depth=depth, lr=lr, steps=steps, batch=100, opt="adam")
    _, H = train_mlp(Xa, ya, Xv, yv, log_every=steps, **hp); acc = 1 - H[-1, 2]
    print(hp, f"val acc {100*acc:.1f}%")
    if best is None or acc > best[0]: best = (acc, hp)
my_hparams = best[1]

---
# Part 9 · Capstone: 🧪 from meerkats to molecules
Time to put chapters 2–8 together on a problem closer to home. This is a **toy, synthetic** phase-separation dataset, invented for this exercise and *not* TBio data. Each sample is a protein condition with three features:
- $x_1$ = log₁₀ protein concentration (µM)
- $x_2$ = salt (mM, scaled)
- $x_3$ = a made-up "stickiness" score of the sequence

The label is $y=1$ if droplets formed. Following the usual cartoon of a phase diagram, the saturation concentration rises with salt and falls with stickiness. Some labels are flipped, because the imaging is noisy.

**Your job, using every chapter:**

| step | chapter |
|---|---|
| pick the output distribution, final activation and loss | 5 |
| build a deep ReLU network in PyTorch | 3–4 |
| use He initialization | 7 |
| train with Adam on minibatches | 6 |
| tune on validation, report test once | 8 |

### 🏆 Puzzle 9 ⭐⭐⭐: reach ≥ 88% validation accuracy, then plot the learned phase diagram.

In [ ]:
def make_condensates(n, seed):
    r = np.random.default_rng(seed)
    logc = r.uniform(-1, 2.5, n); salt = r.uniform(0, 1, n); stick = r.uniform(0, 1, n)
    csat = 0.2 + 1.6 * salt ** 1.5 - 1.1 * stick + 0.4 * np.sin(3 * stick * salt)      # log10 c_sat
    y = (logc > csat).astype(np.float32)
    flip = r.uniform(size=n) < 0.06; y[flip] = 1 - y[flip]
    return np.stack([logc, salt, stick], 1).astype(np.float32), y

Xc_tr, yc_tr = make_condensates(3000, 1); Xc_va, yc_va = make_condensates(1000, 2); Xc_te, yc_te = make_condensates(1000, 3)
fig, ax = plt.subplots(figsize=(5.5, 4))
m = Xc_tr[:, 2] > 0.6
ax.scatter(Xc_tr[m, 1], Xc_tr[m, 0], c=yc_tr[m], cmap=ListedColormap([TB["lblue"], TB["green"]]), s=8)
ax.set_xlabel("salt (scaled)"); ax.set_ylabel(r"log$_{10}$ [protein] (µM)"); ax.set_title("Sticky proteins: green = droplets"); plt.show()

In [ ]:
def train_condensate_model(Xtr, ytr, Xva, yva):
    """Build, initialise, and train a model. Return a function predict_proba(X_numpy) -> probabilities (numpy)."""
    # TODO: torch.nn.Sequential(...), kaiming init, BCEWithLogitsLoss, Adam, minibatches
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 9
def _t9():
    if not HAS_TORCH: print("   (torch missing)"); return False
    predict_proba = train_condensate_model(Xc_tr, yc_tr, Xc_va, yc_va)
    va = np.mean((predict_proba(Xc_va) > 0.5) == yc_va); te = np.mean((predict_proba(Xc_te) > 0.5) == yc_te)
    print(f"   validation accuracy {100*va:.1f}%   |   🔒 test (once) {100*te:.1f}%   |   label-noise ceiling ≈ 94%")
    S, C = np.meshgrid(np.linspace(0, 1, 200), np.linspace(-1, 2.5, 200))
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for a, st in zip(axs, [0.1, 0.5, 0.9]):
        grid = np.stack([C.ravel(), S.ravel(), np.full(C.size, st)], 1).astype(np.float32)
        im = a.contourf(S, C, predict_proba(grid).reshape(S.shape), 20, cmap="BuGn", vmin=0, vmax=1)
        a.set_title(f"stickiness = {st}"); a.set_xlabel("salt")
    axs[0].set_ylabel(r"log$_{10}$ [protein]"); plt.colorbar(im, ax=axs, label="P(droplets)"); plt.show()
    return va >= 0.88
check("9", 3, _t9, "A learned phase diagram. Welcome to the intersection of meerkats and molecules.",
      "Standardise inputs, 2-3 hidden layers of 64, BCEWithLogitsLoss, Adam lr 1e-3 to 3e-3, ~30 epochs of batch 64.")

In [ ]:
#@title 🔑 Solution 9 — double-click to peek (have a go first!)
peek('9')
def train_condensate_model(Xtr, ytr, Xva, yva):
    torch.manual_seed(0)
    mu, sd = Xtr.mean(0), Xtr.std(0)
    net = torch.nn.Sequential(torch.nn.Linear(3, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64), torch.nn.ReLU(),
                              torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1))
    for m in net:
        if isinstance(m, torch.nn.Linear): torch.nn.init.kaiming_normal_(m.weight); m.bias.data.zero_()
    X = torch.tensor((Xtr - mu) / sd); Y = torch.tensor(ytr)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3); lossf = torch.nn.BCEWithLogitsLoss()
    for epoch in range(40):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]; opt.zero_grad(); lossf(net(X[idx])[:, 0], Y[idx]).backward(); opt.step()
    def predict_proba(Xn):
        with torch.no_grad(): return torch.sigmoid(net(torch.tensor((Xn - mu) / sd))[:, 0]).numpy()
    return predict_proba

---
# 🏁 Your progress

In [ ]:
progress()

### ⏭️ Next time
**Next up · Chapter 9 (Regularization):** why do over-parameterised networks generalise at all?

### 📚 Where to go next
- The book's own notebooks ([udlbook GitHub](https://github.com/udlbook/udlbook)) mirror every section here. Try Notebooks 3.3 (regions), 6.5 (Adam), 7.2 (backprop) and 8.3 (double descent).
- MNIST-1D (Greydanus, 2020) is the real version of our meerkat call dataset: `pip install mnist1d`.
- Discussion prompts for the group:
  1. Our condensate model is a flexible classifier. Where would *bias* versus *variance* dominate on real phase-separation data with ~100 measured proteins?
  2. Which output distribution (Part 5) suits a *condensate count per cell*? A *droplet size*? A *partition coefficient*?
  3. Double descent says "bigger can be better". When is that **not** true for our assay data?

*Built for the Transition Bio ML reading group · based on Prince, Understanding Deep Learning (MIT Press, CC-BY-NC-ND) · all data synthetic.*